### Importación de librerías


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import graphviz
from tabulate import tabulate
from itertools import product

# Modelos de Machine Learning
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import xgboost as xgb

# Preprocesamiento
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Model Selection y Validación
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, validation_curve
)

# Métricas
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)

### Cargar datos

In [2]:
df_limpio = pd.read_csv('../data/stg/IA_PROPENSITY_TRAIN_1.csv', index_col=0) 
df_limpio.head()

,TIPO_CARROCERIA,COMBUSTIBLE,Potencia,TRANS,FORMA_PAGO,ESTADO_CIVIL,GENERO,OcupaciOn,PROVINCIA,Campanna1,Campanna2,Campanna3,Zona_Renta,REV_Garantia,Averia_grave,QUEJA_CAC,COSTE_VENTA,km_anno,Mas_1_coche,Revisiones,Edad_Cliente,Tiempo
PRODUCTO,,,,,,,,,,,,,,,,,,,,,,
0,0,0,1,1,0,0,1,1,4,1,0,0,2,0,2,1,2892,0,0,2,18,0
0,0,0,1,1,0,0,0,1,47,0,0,0,2,1,3,0,1376,7187,0,2,53,0
0,0,0,1,1,3,0,1,1,30,0,0,0,1,0,3,0,1376,0,1,4,21,3
0,0,0,1,1,2,0,0,1,32,1,0,0,1,1,2,1,2015,7256,1,4,48,5
0,0,0,1,1,2,0,0,2,41,1,0,1,0,0,3,0,1818,0,1,3,21,3


In [3]:
X = df_limpio.drop(columns=['Tiempo', 'Mas_1_coche']) 
y = df_limpio['Mas_1_coche']  # Objetivo

# Dividir los datos en entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Random Forest

In [ ]:
# Definir los parámetros de los modelos

param_grid = {
    'criterion': ['entropy'],
    'n_estimators': [500],
    'max_depths' : [2, 5],
    'min_samples_split': [6, 8],
    'min_samples_leaf': [4, 7]
}

# Almacenar los resultados

results = []

# Iterar sobre los parámetros

for params in product(*param_grid.values()):
    criterion, n_estimators, max_depth, min_samples_split, min_samples_leaf = params
    model = RandomForestClassifier(
        criterion=criterion,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight='balanced',
        random_state=42
    )

    # Entrenar el modelo

    model.fit(X_train, y_train)

    # Predecir el modelo en los datos de prueba

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.4).astype(int)

    # Calcular las métricas

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Validación cruzada

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    mean_cv_score = np.mean(cv_scores)
    
    # Guardar los resultados

    results.append({
        'criterion': criterion,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'n_estimators': n_estimators,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,  # <-- Priorizar recall
        'roc_auc': roc_auc,
        'cv_recall': mean_cv_score  # <-- Guardar recall en validación cruzada
    })


# Convertir los resultados en un DataFrame

results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1_score'], ascending=False)

# Mostrar los resultados

display(results_df.head(5))


,criterion,max_depth,min_samples_split,min_samples_leaf,n_estimators,accuracy,f1_score,recall,roc_auc,cv_recall
4,entropy,20,9,4,500,0.766456,0.776970,0.766456,0.886432,0.748666
6,entropy,20,12,4,500,0.763429,0.774190,0.763429,0.886274,0.755257
5,entropy,20,9,7,500,0.752703,0.764178,0.752703,0.883719,0.776288
7,entropy,20,12,7,500,0.752703,0.764178,0.752703,0.883719,0.776288
2,entropy,15,12,4,500,0.748119,0.759870,0.748119,0.883534,0.787840


### XGBoost

In [11]:
# Definir los hiperparámetros

param_grid = {
    'learning_rate': [0.05],  # Aumentar el learning_rate
    'n_estimators': [200],  # Aumentar el número de estimadores
    'max_depth': [18],  # Aumentar la profundidad de los árboles
    'min_samples_split': [13],  # Ajustar min_child_weight para más flexibilidad
    'min_samples_leaf': [3],  # Ajustar min_child_weight para más flexibilidad
}

"""'subsample': [0.8],  # Probar un valor más bajo de subsample
    'colsample_bytree': [0.7],  # Ajustar el número de características usadas
    'gamma': [0.1],  # Mantener gamma bajo
    'scale_pos_weight': [10],  # Aumentar scale_pos_weight para clases desbalanceadas
    'base_score': [0.5],  # Mantener base_score en 0.5
    'random_state': [42],  # Semilla fija para reproducibilidad"""



# Almacenar los resultados

results = []

# Iterar sobre los parámetros

for params in product(*param_grid.values()):
    learning_rate, n_estimators, max_depth, min_samples_leaf, min_samples_split = params
    model = XGBClassifier(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        objective='binary:logistic',
        eval_metric='logloss'
    )

    # Entrenar el modelo

    model.fit(X_train, y_train)

    # Predecir el modelo en los datos de prueba

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.4).astype(int)

    # Calcular las métricas

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Validación cruzada

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    mean_cv_score = np.mean(cv_scores)
    
    # Guardar los resultados

    results.append({
        'learning_rate': learning_rate,
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,  # <-- Priorizar recall
        'roc_auc': roc_auc,
        'cv_recall': mean_cv_score  # <-- Guardar recall en validación cruzada
    })


# Convertir los resultados en un DataFrame

results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1_score'], ascending=False)

# Mostrar los resultados

display(results_df.head(5))

c:\Users\1cnac\OneDrive\Documentos\INGENIERÍA MATEMÁTICA\Inteligencia Artificial\ia_trabajos\env\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:26:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\1cnac\OneDrive\Documentos\INGENIERÍA MATEMÁTICA\Inteligencia Artificial\ia_trabajos\env\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:26:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "min_samples_leaf", "min_samples_split" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\1cnac\OneDrive\Documentos\INGENIERÍA MATEMÁTICA\Inteligencia Artificial\ia_trabajos\env\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:26:55] WARNING: C:\buildk

,learning_rate,n_estimators,max_depth,min_samples_split,min_samples_leaf,accuracy,f1_score,recall,roc_auc,cv_recall
0,0.05,200,18,3,13,0.816884,0.817262,0.816884,0.893029,0.618409


### AdaBoost

In [ ]:
param_grid = {
    'n_estimators': [500],
    'base_estimator': [DecisionTreeClassifier(max_depth=8), DecisionTreeClassifier(max_depth=7)],
    'learning_rate': [0.5, 0.6, 0.7], 
    'algorithm': ['SAMME'],
    'random_state': [42]
}

# Almacenar los resultados

results = []

# Iterar sobre los parámetros

for params in product(*param_grid.values()):
    n_estimators, base_estimator, learning_rate, algorithm, random_state = params

    model = AdaBoostClassifier(
        n_estimators=n_estimators,
        base_estimator=base_estimator,
        learning_rate=learning_rate,
        algorithm=algorithm,
        random_state=random_state
    )

     # Entrenar el modelo
    model.fit(X_train, y_train)

    # Predecir el modelo en los datos de prueba
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.4).astype(int)  # Ajustar el umbral de decisión

    # Calcular las métricas
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Validación cruzada
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    mean_cv_score = np.mean(cv_scores)
    
    # Guardar los resultados
    results.append({
        'n_estimators': n_estimators,
        'base_estimator_max_depth': base_estimator.max_depth,
        'learning_rate': learning_rate,
        'algorithm': algorithm,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,  # Priorizar recall
        'roc_auc': roc_auc,
        'cv_recall': mean_cv_score  # Guardar recall en validación cruzada
    })

# Convertir los resultados en un DataFrame y ordenar
results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1_score'], ascending=False)

# Mostrar los mejores resultados
display(results_df.head(5))

### Gradient Boosting

In [ ]:
# Definir los parámetros de Gradient Boosting

param_grid = {
    'loss': ['deviance', 'exponential'],  # Función de pérdida ('deviance' es log-loss, 'exponential' es AdaBoost)
    'learning_rate': [0.01, 0.1, 0.2],  # Tasa de aprendizaje
    'n_estimators': [100, 500],  # Número de estimadores (árboles)
    'subsample': [0.8, 1.0],  # Fracción de muestras usadas en cada árbol (submuestreo)
    'criterion': ['friedman_mse', 'mse'],  # Criterio de división (friedman_mse y mse son comunes)
    'min_samples_split': [2, 5],  # Mínimo número de muestras necesarias para dividir un nodo
    'min_samples_leaf': [1, 5],  # Mínimo número de muestras requeridas en una hoja
    'min_weight_fraction_leaf': [0.0, 0.1],  # Fracción mínima de peso para una hoja
    'max_depth': [3, 5, 10],  # Profundidad máxima de los árboles
    'min_impurity_decrease': [0.0, 0.1],  # Reducción mínima de la impureza para dividir un nodo
    'min_impurity_split': [1e-7],  # Umbral para dividir un nodo
    'init': [None],  # Establece el estimador inicial (None usa el valor por defecto)
    'random_state': [42],  # Fijar la semilla para reproducibilidad
    'max_features': [None, 'sqrt', 'log2'],  # Número máximo de características a usar
    'verbose': [0, 1],  # Nivel de verbosidad
    'warm_start': [False, True]  # Reutilizar el modelo anterior para incrementar estimadores
}

# Almacenar los resultados
results = []

# Iterar sobre los parámetros
for params in product(*param_grid.values()):
    loss, learning_rate, n_estimators, subsample, criterion, min_samples_split, min_samples_leaf, min_weight_fraction_leaf, \
    max_depth, min_impurity_decrease, min_impurity_split, init, random_state, max_features, verbose, warm_start = params
    
    # Crear el modelo Gradient Boosting con los parámetros proporcionados
    model = GradientBoostingClassifier(
        loss=loss,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        criterion=criterion,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        min_weight_fraction_leaf=min_weight_fraction_leaf,
        max_depth=max_depth,
        min_impurity_decrease=min_impurity_decrease,
        min_impurity_split=min_impurity_split,
        init=init,
        random_state=random_state,
        max_features=max_features,
        verbose=verbose,
        warm_start=warm_start
    )

    # Entrenar el modelo
    model.fit(X_train, y_train)

    # Predecir el modelo en los datos de prueba
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba > 0.4).astype(int)  # Ajustar el umbral de decisión

    # Calcular las métricas
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Validación cruzada
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    mean_cv_score = np.mean(cv_scores)
    
    # Guardar los resultados
    results.append({
        'loss': loss,
        'learning_rate': learning_rate,
        'n_estimators': n_estimators,
        'subsample': subsample,
        'criterion': criterion,
        'min_samples_split': min_samples_split,
        'min_samples_leaf': min_samples_leaf,
        'min_weight_fraction_leaf': min_weight_fraction_leaf,
        'max_depth': max_depth,
        'min_impurity_decrease': min_impurity_decrease,
        'min_impurity_split': min_impurity_split,
        'init': init,
        'random_state': random_state,
        'max_features': max_features,
        'verbose': verbose,
        'warm_start': warm_start,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,  # Priorizar recall
        'roc_auc': roc_auc,
        'cv_recall': mean_cv_score  # Guardar recall en validación cruzada
    })

# Convertir los resultados en un DataFrame y ordenar
results_df = pd.DataFrame(results).sort_values(by=['recall', 'f1_score'], ascending=False)

# Mostrar los mejores resultados
display(results_df.head(5))